In [9]:
!pip install gymnasium torch numpy pygame

  Using cached pygame-2.6.1.tar.gz (14.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: still running...
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  Getting requirements to build wheel did not run successfully.
  exit code: 1
  
  [112 lines of output]
  Skipping Cython compilation
  
  
  WARNING, No "Setup" File Exists, Running "buildconfig/config.py"
  Using WINDOWS configuration...
  
  Making dir :prebuilt_downloads:
  Downloading... https://www.libsdl.org/release/SDL2-devel-2.28.4-VC.zip 25ef9d201ce3fd5f976c37dddedac36bd173975c
  Unzipping :prebuilt_downloads\SDL2-devel-2.28.4-VC.zip:
  Downloading... https://www.libsdl.org/projects/SDL_image/release/SDL2_image-devel-2.0.5-VC.zip 137f86474691f4e12e76e07d58d5920c8d844d5b
  Unzipping :prebuilt_downloads\SDL2_image-devel-2.0.5-VC.zip:
  Downloading... https://github.com/libsdl-org/SDL_ttf/releases/download/release-2.20.1/SDL2_ttf-devel-2.20.1-VC.zip 371606aceba450384428fd2852f73d2f6290b136
  Unzipping :prebuilt_downloads\SDL2_ttf-devel-2.20.1-VC.zip:
  Downloading... https://github.com/libsdl-org/SDL_mixer/releases/download/release-2.6.

In [10]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym

# ---------- Setup ----------
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False)
N_STATES, N_ACTIONS = 16, 4
TERMINAL = {5, 7, 11, 12, 15}  # holes + goal

def one_hot(s):
    v = np.zeros(N_STATES, dtype=np.float32)
    v[s] = 1.0
    return v

In [11]:
# ---------- 1. Collect imperfect dataset ----------
def behavior_policy():
    if np.random.rand() < 0.3:
        return env.action_space.sample()  # 30% random
    return np.random.choice([1, 2])       # 70% Down/Right only

S, A, R, S2, D = [], [], [], [], []

for _ in range(300):
    s, _ = env.reset()
    done = False
    while not done:
        a = behavior_policy()
        s2, r, term, trunc, _ = env.step(a)
        done = term or trunc
        S.append(s)
        A.append(a)
        R.append(r)
        S2.append(s2)
        D.append(float(term))
        s = s2

S = np.array(S)
A = np.array(A)
S2 = np.array(S2)
R = np.array(R, dtype=np.float32)
D = np.array(D, dtype=np.float32)

# Split (state, action) pairs into common vs rare
pair_counts = np.zeros((N_STATES, N_ACTIONS), dtype=int)
for s, a in zip(S, A):
    pair_counts[s, a] += 1

valid = [s for s in range(N_STATES) if s not in TERMINAL]
common = [(s, a) for s in valid for a in range(4) if pair_counts[s, a] >= 5]
rare = [(s, a) for s in valid for a in range(4) if pair_counts[s, a] < 5]

In [12]:
# ---------- 2. Q-network ----------
class QNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_STATES, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, N_ACTIONS)
        )

    def forward(self, x):
        return self.net(x)

In [13]:
# ---------- 3. Loss: Bellman + alpha * CQL penalty ----------
def cql_loss(q_net, target_net, s, a, r, s2, done, gamma, alpha):
    q_all = q_net(s)
    q_data = q_all.gather(1, a.unsqueeze(1)).squeeze(1)
    
    with torch.no_grad():
        target = r + gamma * (1 - done) * target_net(s2).max(dim=1).values
    
    bellman = F.mse_loss(q_data, target)
    cql_pen = (torch.logsumexp(q_all, dim=1) - q_data).mean()
    
    return bellman + alpha * cql_pen

In [14]:
# ---------- 4. Offline training ----------
S_T = torch.tensor(np.array([one_hot(x) for x in S]))
S2_T = torch.tensor(np.array([one_hot(x) for x in S2]))
A_T = torch.tensor(A, dtype=torch.long)
R_T = torch.tensor(R, dtype=torch.float32)
D_T = torch.tensor(D, dtype=torch.float32)

def train(alpha, steps=3000, batch=64, lr=1e-3, gamma=0.99):
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    
    q, tgt = QNet(), QNet()
    tgt.load_state_dict(q.state_dict())
    opt = torch.optim.Adam(q.parameters(), lr=lr)
    
    for step in range(steps):
        i = np.random.randint(0, len(S), batch)
        loss = cql_loss(q, tgt, S_T[i], A_T[i], R_T[i], S2_T[i], D_T[i], gamma, alpha)
        
        opt.zero_grad()
        loss.backward()
        opt.step()
        
        if step % 100 == 0:
            tgt.load_state_dict(q.state_dict())
            
    return q

In [15]:
# ---------- 5. Helpers ----------
def q_table(q):
    with torch.no_grad():
        return q(torch.tensor(np.array([one_hot(s) for s in range(N_STATES)]))).numpy()

def evaluate(q):
    s, _ = env.reset(seed=SEED)
    total, done, steps = 0.0, False, 0
    while not done and steps < 50:
        with torch.no_grad():
            a = q(torch.tensor(one_hot(s)).unsqueeze(0)).argmax().item()
        s, r, term, trunc, _ = env.step(a)
        done = term or trunc
        total += r
        steps += 1
    return total  # Deterministic environment

def avg_q(qt, pairs):
    return float(np.mean([qt[s, a] for s, a in pairs]))

In [16]:
# ---------- 6. Run and print key results ----------
# Train Offline DQN (alpha = 0.0)
dqn_model = train(alpha=0.0)
dqn_q = q_table(dqn_model)
dqn_ret = evaluate(dqn_model)

# Train CQL (alpha = 1.0)
cql_model = train(alpha=1.0)
cql_q = q_table(cql_model)
cql_ret = evaluate(cql_model)

# Print Summary
print(f"Dataset: {len(S)} transitions | "
      f"Left used {100*np.mean(A==0):.0f}%, "
      f"Down {100*np.mean(A==1):.0f}%, "
      f"Right {100*np.mean(A==2):.0f}%, "
      f"Up {100*np.mean(A==3):.0f}%")

print(f"Rare/unseen (state,action) pairs: {len(rare)} of {len(rare)+len(common)}\n")
print(f"{'':<12}{'Return':>8}{'Avg Q common':>15}{'Avg Q rare':>13}{'Rare Q > 1.0':>15}")

for name, qt, ret in [("Offline DQN", dqn_q, dqn_ret), ("CQL", cql_q, cql_ret)]:
    over = sum(qt[s, a] > 1.0 for s, a in rare)
    print(f"{name:<12}{ret:>8.1f}{avg_q(qt, common):>15.3f}{avg_q(qt, rare):>13.3f}{over:>11d}/{len(rare)}")

print("\nNote: the max possible reward is 1.0, so any Q above 1.0 is overestimation.")

Dataset: 1263 transitions | Left used 7%, Down 43%, Right 44%, Up 6%
Rare/unseen (state,action) pairs: 10 of 44

              Return   Avg Q common   Avg Q rare   Rare Q > 1.0
Offline DQN      1.0          0.796        0.666          0/10
CQL              0.0          1.180        0.008          0/10

Note: the max possible reward is 1.0, so any Q above 1.0 is overestimation.
